# XOR με ένα μικρό νευρωνικό δίκτυο

Ένα σύντομο παράδειγμα PyTorch για Google Colab. Το μοντέλο μαθαίνει τον κανόνα XOR από τέσσερα παραδείγματα.

**XOR** δίνει 1 όταν οι δύο είσοδοι είναι διαφορετικές και 0 όταν είναι ίδιες. Το Colab έχει συνήθως ήδη εγκατεστημένο το PyTorch.

## 1. Τα παραδείγματα

Κάθε γραμμή έχει δύο εισόδους και τη σωστή απάντηση. Το μοντέλο θα προσπαθήσει να μάθει αυτή τη σχέση.

In [1]:
import torch
from torch import nn

torch.manual_seed(7)  # Σταθερή αρχική τυχαιότητα

# Τέσσερις πιθανοί συνδυασμοί εισόδων
inputs = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
answers = torch.tensor([[0.], [1.], [1.], [0.]])

print('Είσοδοι:', inputs.tolist())
print('Σωστές απαντήσεις:', answers.flatten().tolist())

Είσοδοι: [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
Σωστές απαντήσεις: [0.0, 1.0, 1.0, 0.0]


## 2. Το μοντέλο

Το δίκτυο έχει 2 εισόδους, 4 κρυφούς νευρώνες και 1 έξοδο. Το `Tanh` προσθέτει μη γραμμικότητα, που χρειάζεται για να μάθει το XOR.

In [2]:
model = nn.Sequential(
    nn.Linear(2, 4),  # Είσοδοι προς κρυφό επίπεδο
    nn.Tanh(),         # Μη γραμμική ενεργοποίηση
    nn.Linear(4, 1)    # Κρυφό επίπεδο προς έξοδο
)

loss_fn = nn.BCEWithLogitsLoss()  # Μετρά το σφάλμα
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Αλλάζει τα βάρη

## 3. Εκπαίδευση

Σε κάθε βήμα το μοντέλο προβλέπει, υπολογίζει το σφάλμα και ενημερώνει τα βάρη του. Το `backward()` υπολογίζει τις διορθώσεις· το `step()` τις εφαρμόζει.

In [3]:
loss_history = []

for step in range(1000):
    optimizer.zero_grad()          # Καθαρίζει παλιές διορθώσεις
    scores = model(inputs)          # Πρόβλεψη για τις εισόδους
    loss = loss_fn(scores, answers)  # Σύγκριση με τις σωστές απαντήσεις
    loss_history.append(loss.item())
    loss.backward()                 # Υπολογίζει διορθώσεις
    optimizer.step()                # Ενημερώνει τα βάρη

    if (step + 1) % 200 == 0:
        print(f'Βήμα {step + 1}: σφάλμα = {loss.item():.4f}')

Βήμα 200: σφάλμα = 0.0019
Βήμα 400: σφάλμα = 0.0007
Βήμα 600: σφάλμα = 0.0004
Βήμα 800: σφάλμα = 0.0003
Βήμα 1000: σφάλμα = 0.0002


## 4. Έλεγχος αποτελεσμάτων

Η έξοδος του μοντέλου μετατρέπεται σε πιθανότητα από 0 έως 1. Με όριο το 0.5 αποφασίζουμε αν η πρόβλεψη είναι 0 ή 1.

In [4]:
model.eval()
with torch.no_grad():  # Δεν χρειάζονται διορθώσεις στην αξιολόγηση
    probabilities = torch.sigmoid(model(inputs))
    predictions = (probabilities >= 0.5).int()

correct = (predictions == answers.int()).sum().item()
accuracy = correct / len(answers)

for pair, answer, probability, prediction in zip(inputs, answers, probabilities, predictions):
    print(f'{pair.tolist()} -> σωστό: {int(answer.item())}, πρόβλεψη: {int(prediction.item())}, πιθανότητα 1: {probability.item():.3f}')

print(f'Ακρίβεια στα 4 παραδείγματα: {correct}/4 ({accuracy:.0%})')

[0.0, 0.0] -> σωστό: 0, πρόβλεψη: 0, πιθανότητα 1: 0.000
[0.0, 1.0] -> σωστό: 1, πρόβλεψη: 1, πιθανότητα 1: 1.000
[1.0, 0.0] -> σωστό: 1, πρόβλεψη: 1, πιθανότητα 1: 1.000
[1.0, 1.0] -> σωστό: 0, πρόβλεψη: 0, πιθανότητα 1: 0.000
Ακρίβεια στα 4 παραδείγματα: 4/4 (100%)


## Τι να θυμάσαι

- Το μοντέλο ξεκινά με βάρη και biases που αρχικά δεν ξέρουν τη λύση.
- Η εκπαίδευση προσαρμόζει αυτά τα νούμερα για να μειώσει το σφάλμα.
- Τα 4/4 σωστά αφορούν μόνο αυτά τα τέσσερα παραδείγματα. Δεν σημαίνουν ότι το μοντέλο καταλαβαίνει γενικά ή λύνει κάθε πρόβλημα.

